# Validation audit

Run an existing stacked BiLSTM checkpoint on full VAL and labeling_sample. Set the checkpoint and its original tokenizer below. Use the same cleanup options as the submitted run. An optional submission CSV checks whether TEST predictions reproduce that file.

## Setup

In [1]:
import os
import sys
import base64
import subprocess
from pathlib import Path

repo_dir = Path("/content/UA-Locations-NER-Deep-Learning")
if not repo_dir.exists():
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = dict(os.environ, GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraheader",
               GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}")
    try:
        subprocess.run(["git", "clone",
                        "https://github.com/dkalchenko01/UA-Locations-NER-Deep-Learning.git",
                        str(repo_dir)], env=env, check=True)
    finally:
        del token, auth, env
sys.path.insert(0, str(repo_dir))


In [2]:
%pip install -q -r "$repo_dir/requirements.txt"

import json
import pandas as pd
import torch
from google.colab import drive
from tokenizers import Tokenizer

from src.data_io import get_data_dir, load_manifest, load_data_splits, file_sha256, save_json
from src.dataset import prepare_ner_records, prepare_inference_records, make_loader
from src.model import BiLSTMTagger
from src.training import predict
from src.metrics import competition_scores, location_errors
from src.postprocessing import postprocess
from src.preprocessing import clean_locations_v3

drive.mount("/content/drive")


Mounted at /content/drive


## Checkpoint and submission settings

Replace both paths. `tokenizer_path` must point to the tokenizer used for this checkpoint. Set `submission_path` to an existing submission CSV to check exact TEST reproduction, or keep `None`. Expansion below applies to spans containing more than four letters.

In [3]:
checkpoint_path = Path("/content/drive/MyDrive/UA-Locations-NER/stacked/stacked_20261112T084900Z_7c1467a9/epoch_03.pt")
tokenizer_path = repo_dir / "artifacts/tokenizers/bpe_bytelevel.json"
submission_path = None
clean_options = {"expand": True, "trim": True, "empty": True}

splits_dir = repo_dir / "data/splits"
output_dir = checkpoint_path.parent / f"audit_{checkpoint_path.stem}"
output_dir.mkdir(exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
config = checkpoint["config"]
provenance = checkpoint.get("provenance", {})
for path, key in [(tokenizer_path, "tokenizer_sha256"),
                  (splits_dir / "validation_manifest.parquet", "manifest_sha256")]:
    if provenance.get(key) and file_sha256(path) != provenance[key]:
        raise ValueError(f"Checkpoint provenance mismatch: {path}")
tokenizer = Tokenizer.from_file(str(tokenizer_path))
tokenizer.no_padding()
tokenizer.no_truncation()
model = BiLSTMTagger(**checkpoint["model_config"]).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
loader_options = {"pad_id": tokenizer.token_to_id("[PAD]"),
                  "batch_size": config["batch_size"],
                  "max_padded_tokens": config["max_padded_tokens"]}
print(device, checkpoint_path, "epoch", checkpoint["epoch"])


cpu /content/drive/MyDrive/UA-Locations-NER/stacked/stacked_20261112T084900Z_7c1467a9/epoch_03.pt epoch 3


## Full validation and labeling sample

Keep every validation row, including examples without aligned BIO labels. These examples contribute to F1 and are excluded from token loss.

In [4]:
data_dir = get_data_dir("colab")
manifest = load_manifest(splits_dir)
val_records, alignment = [], []
for source in config["sources"]:
    splits = load_data_splits("colab", splits_dir, sources=(source,))
    records, skipped = prepare_ner_records(tokenizer, splits[f"{source}_val"], source, "val")
    val_records.extend(records)
    alignment.append({"source": source, "rows": len(records), "unaligned": len(skipped)})
    del splits
expected = manifest[
    (manifest["split"] == "val") & manifest["source"].isin(config["sources"])]
assert {r["text_id"] for r in val_records} == {
    f"{row.source}_{row.row_id}" for row in expected.itertuples()}
sample = pd.read_csv(data_dir / "labeling_sample.csv")
sample_records, _ = prepare_ner_records(tokenizer, sample, "sample", "val")
display(pd.DataFrame(alignment))


100%|██████████| 811M/811M [00:10<00:00, 79.4MB/s]

Extracting files...


Prepare uk/val:   0%|          | 0/51 [00:00<?, ?it/s]

Prepare ru/val:   0%|          | 0/13 [00:00<?, ?it/s]

Prepare sample/val:   0%|          | 0/1 [00:00<?, ?it/s]

,source,rows,unaligned
0,uk,100741,165
1,ru,24626,23


## Compare decoding and cleanup

`raw` uses decoded spans, `clean_v3` applies the previous cleaner, and `submission` uses the options above. All scores use the existing competition metric. Token loss is independent of decoding and cleanup.

In [5]:
datasets = {"val": val_records, "sample": sample_records}
scores, losses, sample_predictions = [], {}, {}

def cleaned(records, predictions, mode):
    result = {}
    for record in records:
        prediction = predictions[record["text_id"]]
        locations = prediction["locations"]
        if mode == "clean_v3":
            locations = clean_locations_v3(locations)
        if mode == "submission":
            locations = postprocess(record["text"], prediction["spans"], **clean_options)
        result[record["text_id"]] = {"locations": locations}
    return result

for name, records in datasets.items():
    for decoder in ("argmax", "viterbi_lines"):
        predictions, token_losses = predict(
            model, make_loader(records, **loader_options), device, decoding=decoder)
        losses[name] = token_losses
        for mode in ("raw", "clean_v3", "submission"):
            scores.append({"dataset": name, "decoder": decoder, "cleanup": mode,
                           **competition_scores(records, cleaned(records, predictions, mode))})
        if name == "sample":
            sample_predictions[decoder] = predictions
        del predictions
scores = pd.DataFrame(scores)
display(scores)
display(pd.DataFrame(losses))
scores.to_csv(output_dir / "scores.csv", index=False)
save_json(output_dir / "settings.json", {
    "checkpoint": str(checkpoint_path), "epoch": checkpoint["epoch"],
    "checkpoint_sha256": file_sha256(checkpoint_path),
    "tokenizer_sha256": file_sha256(tokenizer_path),
    "manifest_sha256": file_sha256(splits_dir / "validation_manifest.parquet"),
    "cleanup": clean_options, "token_losses": losses})


Predict:   0%|          | 0/986 [00:00<?, ?it/s]

Predict:   0%|          | 0/986 [00:00<?, ?it/s]

Predict:   0%|          | 0/1 [00:00<?, ?it/s]

Predict:   0%|          | 0/1 [00:00<?, ?it/s]

,dataset,decoder,cleanup,uk_comp_f1,ru_comp_f1,combined_comp_f1,precision,recall,TP,FP,FN,sample_comp_f1
0,val,argmax,raw,0.886456,0.911714,0.892849,0.889378,0.896347,40834,5079,4722,NaN
1,val,argmax,clean_v3,0.886146,0.911714,0.892619,0.889332,0.895930,40815,5079,4741,NaN
2,val,argmax,submission,0.881331,0.917038,0.890375,0.889234,0.891518,40614,5059,4942,NaN
3,val,viterbi_lines,raw,0.890277,0.917838,0.897250,0.897319,0.897181,40872,4677,4684,NaN
4,val,viterbi_lines,clean_v3,0.889928,0.917838,0.896991,0.897217,0.896764,40853,4680,4703,NaN
5,val,viterbi_lines,submission,0.884261,0.920672,0.893483,0.895236,0.891738,40624,4754,4932,NaN
6,sample,argmax,raw,NaN,NaN,0.484211,0.365079,0.718750,23,40,9,0.484211
7,sample,argmax,clean_v3,NaN,NaN,0.574713,0.454545,0.781250,25,30,7,0.574713
8,sample,argmax,submission,NaN,NaN,0.620690,0.490909,0.843750,27,28,5,0.620690
9,sample,viterbi_lines,raw,NaN,NaN,0.585366,0.480000,0.750000,24,26,8,0.585366


,val,sample
uk,0.016521,NaN
ru,0.019449,NaN
combined,0.017255,0.219228
sample,NaN,0.219228


## Read all sample errors

FP/FN use official normalization. Gold spans, decoded spans and original strings remain visible to inspect boundaries. Fill in `comment` in the saved review CSV.

In [6]:
review = []
for record in sample_records:
    text_id = record["text_id"]
    predictions = sample_predictions["viterbi_lines"][text_id]
    locations = postprocess(record["text"], predictions["spans"], **clean_options)
    errors = location_errors(record["gold_locations"], locations)
    review.append({"text_id": text_id, "text": record["text"],
                   "gold": record["gold_locations"], "raw": predictions["locations"],
                   "predicted": locations, **errors, "comment": ""})
    print(text_id, "\n", record["text"])
    print("GOLD:", record["gold_locations"])
    print("GOLD SPANS:", sample.loc[record["row_id"], "loc_markers"])
    print("PRED SPANS:", predictions["spans"], "RAW:", predictions["locations"])
    print("FINAL:", locations, "\nFP:", errors["false_positives"],
          "\nFN:", errors["false_negatives"], "\n")
pd.DataFrame(review).to_csv(output_dir / "sample_review.csv", index=False)


sample_0 
 -261.310
GOLD: []
GOLD SPANS: []
PRED SPANS: [] RAW: []
FINAL: [] 
FP: [] 
FN: [] 

sample_1 
 Проспект Бандери стоїть, як і пів міста… довго ще??

Життя🫶Оболонь
GOLD: ['Проспект Бандери']
GOLD SPANS: [(0, 16)]
PRED SPANS: [(53, 59)] RAW: ['Життя🫶']
FINAL: ['Життя'] 
FP: ['життя'] 
FN: ['проспект бандери'] 

sample_2 
 ☢️Радіаційний фон (мкР/год) по Київщині станом на 9 год 28 серпня:

Чорнобиль - 19
Київ - 12
Тетерів - 9
Фастів - 11
Біла Церква - 13
Миронівка - 13
Бориспіль - 13
Яготин - 12
Баришівка – 10

Всі показники у межах норми ✅

ℹ️ Нагадуємо, що допустиме значення рівня радіаційного фону до 25 мкР/год

Сайт💧Instagram💧Facebook💧YouTube
GOLD: ['Київщині', 'Чорнобиль', 'Київ', 'Тетерів', 'Фастів', 'Біла Церква', 'Миронівка', 'Бориспіль', 'Яготин', 'Баришівка']
GOLD SPANS: [(31, 39), (68, 77), (83, 87), (93, 100), (105, 111), (117, 128), (134, 143), (149, 158), (164, 170), (176, 185)]
PRED SPANS: [(31, 39), (83, 87), (117, 128), (164, 170), (176, 185)] RAW: ['Київщині', 